# Day 4: Sentiment Models and Evaluation

This notebook intentionally stops with an error until Fortune's manual labels are complete. The labels are the evaluation reference and must not be generated automatically.

Core comparison:

- VADER rule-based baseline
- TF-IDF plus Logistic Regression classifier
- Accuracy, macro F1, per-class metrics, confusion matrices, and error analysis

If VADER is missing in Colab, run this once in a separate cell:

```python
%pip install vaderSentiment
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

CLEAN_DIR = PROJECT_DIR / 'data' / 'clean'
FIGURES_DIR = PROJECT_DIR / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LABEL_PATH = CLEAN_DIR / 'articles_to_label.csv'

data = pd.read_csv(LABEL_PATH)
data['sentiment_label'] = data['sentiment_label'].fillna('').str.strip().str.lower()
data['confidence'] = data['confidence'].fillna('').str.strip().str.lower()
data['label_reason'] = data['label_reason'].fillna('').str.strip()

valid_labels = {'positive', 'neutral', 'negative'}
if not data['sentiment_label'].isin(valid_labels).all():
    raise ValueError('Complete every sentiment_label with positive, neutral, or negative before modelling.')
if not data['confidence'].isin({'high', 'medium', 'low'}).all():
    raise ValueError('Complete every confidence value with high, medium, or low before modelling.')
if data['label_reason'].eq('').any():
    raise ValueError('Every label needs a short label_reason before modelling.')

print('Rows ready for modelling:', len(data))
display(data['sentiment_label'].value_counts().to_frame('count'))

In [ ]:
if data['sentiment_label'].value_counts().min() < 2:
    raise ValueError('Each class needs at least two examples for a stratified split.')

train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data['sentiment_label'],
)
print('Training rows:', len(train_data))
print('Test rows:', len(test_data))

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_label(text):
    compound = analyzer.polarity_scores(text)['compound']
    if compound >= 0.05:
        return 'positive'
    if compound <= -0.05:
        return 'negative'
    return 'neutral'

test_data = test_data.copy()
test_data['vader_prediction'] = test_data['headline'].map(vader_label)
vader_report = classification_report(
    test_data['sentiment_label'],
    test_data['vader_prediction'],
    labels=['negative', 'neutral', 'positive'],
    output_dict=True,
    zero_division=0,
)
print('VADER baseline report:')
display(pd.DataFrame(vader_report).transpose())

In [ ]:
text_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1),),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

text_model.fit(train_data['headline'], train_data['sentiment_label'])
test_data['model_prediction'] = text_model.predict(test_data['headline'])

model_report = classification_report(
    test_data['sentiment_label'],
    test_data['model_prediction'],
    labels=['negative', 'neutral', 'positive'],
    output_dict=True,
    zero_division=0,
)
print('TF-IDF plus Logistic Regression report:')
display(pd.DataFrame(model_report).transpose())

In [ ]:
comparison = pd.DataFrame([
    {
        'model': 'VADER',
        'accuracy': accuracy_score(test_data['sentiment_label'], test_data['vader_prediction']),
        'macro_f1': vader_report['macro avg']['f1-score'],
    },
    {
        'model': 'TF-IDF + Logistic Regression',
        'accuracy': accuracy_score(test_data['sentiment_label'], test_data['model_prediction']),
        'macro_f1': model_report['macro avg']['f1-score'],
    },
])
comparison_path = CLEAN_DIR / 'model_comparison.csv'
comparison.to_csv(comparison_path, index=False)
display(comparison)
print('Comparison saved:', comparison_path)

In [ ]:
labels = ['negative', 'neutral', 'positive']

for name, prediction_column, filename in [
    ('VADER', 'vader_prediction', 'confusion_matrix_vader.png'),
    ('TF-IDF + Logistic Regression', 'model_prediction', 'confusion_matrix_logistic_regression.png'),
]:
    figure, axis = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay.from_predictions(
        test_data['sentiment_label'],
        test_data[prediction_column],
        labels=labels,
        display_labels=labels,
        cmap='Blues',
        ax=axis,
    )
    axis.set_title(name)
    figure.tight_layout()
    figure.savefig(FIGURES_DIR / filename, dpi=150)
    plt.show()

In [ ]:
errors = test_data[
    (test_data['sentiment_label'] != test_data['model_prediction'])
][
    ['row_id', 'headline', 'sentiment_label', 'model_prediction', 'label_reason']
]
error_path = CLEAN_DIR / 'model_error_analysis.csv'
errors.to_csv(error_path, index=False)
print('Model errors:', len(errors))
print('Error analysis saved:', error_path)
display(errors.head(20))

In [ ]:
data['model_sentiment'] = text_model.predict(data['headline'])
score_map = {'negative': -1, 'neutral': 0, 'positive': 1}
data['model_sentiment_score'] = data['model_sentiment'].map(score_map)
data['pub_datetime'] = pd.to_datetime(data['pub_datetime'], utc=True)
data['week'] = data['pub_datetime'].dt.tz_localize(None).dt.to_period('W').astype(str)

weekly_sentiment = (
    data.groupby('week')['model_sentiment']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)
weekly_sentiment_path = CLEAN_DIR / 'weekly_sentiment_summary.csv'
weekly_sentiment.to_csv(weekly_sentiment_path, index=False)

predictions_path = CLEAN_DIR / 'articles_with_model_predictions.csv'
data.to_csv(predictions_path, index=False)
display(weekly_sentiment)
print('Weekly sentiment saved:', weekly_sentiment_path)
print('Predictions saved:', predictions_path)

## Interpretation rules

Do not choose a model only because it has higher accuracy. Discuss macro F1, class balance, confusion patterns, and representative errors. The resulting trend describes predicted news-headline sentiment, not the opinions of all Nigerians.